# Multimodal AI — Hands-On

Offline lab: toy CLIP-style alignment and multimodal retrieval over text, image, and audio vectors.

## 0. Setup

In [ ]:
%pip install -q numpy
import numpy as np

## 1. CLIP-style cosine alignment

In [ ]:
def normalize(x):
    x = np.asarray(x, dtype=float)
    return x / (np.linalg.norm(x) + 1e-12)
def cosine(a, b): return float(np.dot(normalize(a), normalize(b)))
image_vecs = {"dog_photo": [0.9, 0.1, 0.2], "chart": [0.1, 0.9, 0.3]}
text_vecs = {"a dog running": [0.85, 0.05, 0.25], "a revenue chart": [0.05, 0.95, 0.2]}
for img, iv in image_vecs.items():
    best = max(text_vecs, key=lambda t: cosine(iv, text_vecs[t]))
    print(img, '->', best, round(cosine(iv, text_vecs[best]), 3))
assert cosine([1,0],[1,0]) > 0.99

## 2. Contrastive score matrix

In [ ]:
I = np.array([[.9,.1,.2],[.1,.9,.3]])
T = np.array([[.85,.05,.25],[.05,.95,.2]])
I = I / np.linalg.norm(I, axis=1, keepdims=True)
T = T / np.linalg.norm(T, axis=1, keepdims=True)
print(np.round(I @ T.T, 3))

## 3. Multimodal RAG ranking

In [ ]:
def rank_multimodal(query, items, weights=None):
    weights = weights or {"text":0.5, "image":0.5, "audio":0.0}
    q = {k: np.asarray(v, float) for k, v in query.items()}
    scored=[]
    for item in items:
        score=0.0
        for mod,w in weights.items():
            if mod in q and mod in item:
                a=q[mod]/(np.linalg.norm(q[mod])+1e-12); b=np.asarray(item[mod],float); b=b/(np.linalg.norm(b)+1e-12)
                score += w*float(np.dot(a,b))
        scored.append((score,item['id']))
    return sorted(scored, reverse=True)
items=[{'id':'slide_dog','text':[.8,.1],'image':[.9,.1]},{'id':'slide_sales','text':[.1,.9],'image':[.2,.8]}]
print(rank_multimodal({'text':[.7,.2], 'image':[1,0]}, items))
assert rank_multimodal({'text':[.7,.2], 'image':[1,0]}, items)[0][1] == 'slide_dog'

## 4. Toy STT and document VQA routing

In [ ]:
def transcribe(audio_id): return {'clip1':'revenue increased', 'clip2':'dog bark'}[audio_id]
def route_question(q): return 'document_vqa' if any(w in q.lower() for w in ['figure','image','chart','page']) else 'text_qa'
print(transcribe('clip1'))
print(route_question('What does the chart show on page 2?'))

## 5. Exercises and links
1. Change modality weights and observe ranking flips.
2. Add captions as a second text field.
3. Simulate audio embeddings plus transcripts.

Cross-link: [[02 Literature Notes/LLM Engineering/Embeddings]].